# 02 · RQ2: single- vs multi-source fine-tuning and OOD generalization (P3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/02_rq2_generalization.ipynb)

Holds the base model, LoRA configuration and **total training tokens (1M)** fixed and compares

* `single`: SQuAD only
* `multi`: SQuAD + NewsQA + TriviaQA (equal token share)

over **3 seeds** (data sample + LoRA init). Evaluation on ID sets and the unseen BioASQ and DuoRC
sets with seed mean ± std, paired bootstrap CIs over questions, and an error analysis.
The multi-source seed-0 run is shared with RQ1. Runtime on a T4: ~45 min.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
GPU_PROFILE = "auto"  #@param ["auto", "t4", "a100"]
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if any(k in GPU for k in ("A100", "H100")) else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"  # different budgets/sizes: keep results of the two profiles apart
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} | profile: {GPU_PROFILE} | outputs -> {OUT}")
print("configs:", CONFIG_FILES)

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
!python -m cegqa.cli rq2 $CONFIG

## Statistical comparison

In [ ]:
from cegqa.utils import load_config
cfg = load_config(CONFIG_FILES)

In [ ]:
import json, pandas as pd
st = json.load(open(f"{cfg['output_dir']}/analysis/rq2_stats.json"))
base, treat = list(st["conditions"])
rows = []
for src, e in st["by_source"].items():
    rows.append({"eval set": src, "OOD": e["ood"],
                 f"F1 {base}": f"{e[base]['f1']['mean']:.1f} ± {e[base]['f1']['std']:.1f}",
                 f"F1 {treat}": f"{e[treat]['f1']['mean']:.1f} ± {e[treat]['f1']['std']:.1f}",
                 "ΔF1": round(e["paired_f1"]["diff"], 2),
                 "95% CI": f"[{e['paired_f1']['lo']:+.1f}, {e['paired_f1']['hi']:+.1f}]",
                 "p (2-sided)": round(e["paired_f1"]["p_two_sided"], 4),
                 "ΔEM": round(e["paired_em"]["diff"], 2)})
pd.DataFrame(rows)

## Error analysis

In [ ]:
cats = []
for src, e in st["by_source"].items():
    for cond in (base, treat):
        cats.append({"eval set": src, "condition": cond, **{k: round(100 * v, 1) for k, v in e[cond]["errors"]["categories"].items()}})
display(pd.DataFrame(cats))
for src, e in st["by_source"].items():
    if e["ood"]:
        print(f"\n== {src}: questions where the two conditions disagree ==")
        display(pd.DataFrame(e["examples"]))

In [ ]:
!python -m cegqa.cli report $CONFIG
from IPython.display import Image
Image(f"{cfg['output_dir']}/figures/rq2_f1_bars.png")